# BreatheSafe - Complete Pipeline
Smog & Asthma Risk Predictor - NUML Machine Learning Lab (Open Ended Lab), Fall 2026

**Group leader:** Abdul Hannan Khan (RC-623) | **Members:** Muhammad Ali Khan (RC-629), Hafsa Qureshi (RC-617)

Run all cells top to bottom: Data Collection → EDA & Plots → Feature Engineering → 3 ML Models → Statistical Evaluation → Live Test Prediction. Same code as scripts 01-03.

# 01 - Data Collection & Merge
BreatheSafe - Smog & Asthma Risk Predictor (NUML, ML Lab, Fall 2026)

This script collects four datasets and merges them into one modelling table:
1. Global Air Pollution Dataset (Kaggle)
2. Air Quality Data in India (Kaggle)
3. Asthma Disease Dataset (Kaggle)
4. Open-Meteo Air Quality + Weather API (live, no key needed)

If a Kaggle CSV is not in `data/raw/`, or the API cannot be reached, a realistic
fallback generator is used so the pipeline always runs. Every fallback is logged
and recorded in `outputs/metrics/dataset_summary.json`.

In [ ]:
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import requests

warnings.filterwarnings("ignore")

try:
    HERE = Path(__file__).resolve().parent
except NameError:  # running inside Jupyter
    HERE = Path.cwd()
ROOT = HERE.parent if HERE.name == "code" else HERE
RAW, PROC, METRICS = ROOT / "data/raw", ROOT / "data/processed", ROOT / "outputs/metrics"
for p in (RAW, PROC, METRICS, ROOT / "outputs/plots", ROOT / "models"):
    p.mkdir(parents=True, exist_ok=True)

SEED = 42
rng = np.random.default_rng(SEED)
summary = {}  # filled as we go, saved at the end

# city: (lat, lon, base PM2.5 for fallback, base temperature for fallback, population weight)
CITIES = {
    "Lahore": (31.5497, 74.3436, 62, 31, 0.30),
    "Islamabad": (33.6844, 73.0479, 38, 28, 0.15),
    "Rawalpindi": (33.5651, 73.0169, 44, 29, 0.15),
    "Karachi": (24.8607, 67.0011, 30, 31, 0.28),
    "Peshawar": (34.0151, 71.5249, 55, 30, 0.12),
}
AQ_URL = "https://air-quality-api.open-meteo.com/v1/air-quality"
WX_URL = "https://api.open-meteo.com/v1/forecast"

## Dataset 4 - Open-Meteo live air quality + weather (5 Pakistani cities)
Up to 92 past days of hourly data per city, so five cities give roughly 11,000 rows.

In [ ]:
def fetch_open_meteo(city, lat, lon, past_days=92):
    common = dict(latitude=lat, longitude=lon, past_days=past_days, forecast_days=1, timezone="Asia/Karachi")
    aq = requests.get(AQ_URL, params={**common, "hourly": "pm10,pm2_5,nitrogen_dioxide,sulphur_dioxide,ozone,carbon_monoxide"}, timeout=30)
    aq.raise_for_status()
    wx = requests.get(WX_URL, params={**common, "hourly": "temperature_2m,relative_humidity_2m,wind_speed_10m"}, timeout=30)
    wx.raise_for_status()
    df = pd.DataFrame(aq.json()["hourly"]).merge(pd.DataFrame(wx.json()["hourly"]), on="time")
    df = df.rename(columns={"time": "timestamp", "nitrogen_dioxide": "no2", "sulphur_dioxide": "so2", "ozone": "o3",
                            "carbon_monoxide": "co", "temperature_2m": "temp", "relative_humidity_2m": "humidity",
                            "wind_speed_10m": "wind"})
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df["city"] = city
    return df[df["timestamp"] <= pd.Timestamp.now()]  # drop forecast hours


def synthetic_timeline():
    """Fallback: hourly smog + weather for 5 cities, Sep-Nov 2025, smog building toward November."""
    idx = pd.date_range("2025-09-01", periods=2208, freq="h")
    n = len(idx)
    hod, prog = idx.hour.values, np.arange(n) / n
    frames = []
    for city, (lat, lon, base, tbase, _) in CITIES.items():
        season = 0.55 + 1.9 * prog ** 1.6
        diurnal = 1 + 0.28 * np.cos((hod - 5) / 24 * 2 * np.pi)
        ar = np.zeros(n)
        for i in range(1, n):
            ar[i] = 0.96 * ar[i - 1] + rng.normal(0, 0.09)
        episode = np.zeros(n)
        for s in rng.choice(n - 100, 8, replace=False):
            episode[s:s + rng.integers(36, 96)] += rng.uniform(0.3, 0.9)
        pm25 = base * season * diurnal * np.exp(ar + 0.6 * episode)
        cooling = 5 if city == "Karachi" else 12
        temp = tbase - cooling * prog + 7 * np.sin((hod - 9) / 24 * 2 * np.pi) + rng.normal(0, 1.2, n)
        hum = np.clip(55 + 20 * prog - 1.2 * (temp - 22) + rng.normal(0, 6, n), 10, 100)
        wind = rng.gamma(2.0, 4.0, n) * np.where(episode > 0.3, 0.5, 1.0)
        df = pd.DataFrame({
            "timestamp": idx, "city": city, "pm2_5": pm25,
            "pm10": pm25 * rng.uniform(1.5, 2.3, n) + rng.normal(0, 6, n),
            "no2": 18 + 0.12 * pm25 + rng.normal(0, 5, n),
            "so2": 6 + 0.05 * pm25 + rng.normal(0, 2, n),
            "o3": np.clip(40 + 35 * np.sin((hod - 9) / 24 * 2 * np.pi) - 0.05 * pm25 + rng.normal(0, 8, n), 5, None),
            "co": 300 + 3 * pm25 + rng.normal(0, 40, n),
            "temp": temp, "humidity": hum, "wind": wind,
        })
        # typical field faults: negative readings, spikes, and gaps
        cols = ["pm2_5", "pm10", "no2", "so2", "o3", "co", "temp", "humidity", "wind"]
        neg = rng.choice(n, int(0.003 * n), replace=False)
        df.loc[neg, "pm2_5"] *= -1
        spk = rng.choice(n, int(0.005 * n), replace=False)
        df.loc[spk, "pm2_5"] *= 6
        df.loc[spk, "pm10"] *= 3
        for _ in range(4):
            s = rng.integers(30, n - 30)
            df.loc[s:s + rng.integers(3, 10), cols] = np.nan
        for c in cols:
            df.loc[rng.random(n) < 0.005, c] = np.nan
        frames.append(df)
    return pd.concat(frames, ignore_index=True)


try:
    timeline = pd.concat([fetch_open_meteo(c, v[0], v[1]) for c, v in CITIES.items()], ignore_index=True)
    if len(timeline) < 10000:
        raise ValueError(f"only {len(timeline)} live rows")
    ds4_mode = "live Open-Meteo API"
except Exception as exc:  # no internet, API down, firewall...
    print(f"[WARN] Open-Meteo unavailable ({type(exc).__name__}: {exc}). Using synthetic Open-Meteo-style timeline.")
    timeline = synthetic_timeline()
    ds4_mode = "FALLBACK synthetic (Open-Meteo format)"

timeline = timeline.sort_values(["city", "timestamp"]).reset_index(drop=True)
timeline.to_csv(PROC / "environment_timeline.csv", index=False)
print(f"Dataset 4 [{ds4_mode}]: {len(timeline):,} hourly records, {timeline.city.nunique()} cities")
summary["dataset_4_open_meteo"] = {"records": int(len(timeline)), "mode": ds4_mode}

## Dataset 2 - Air Quality Data in India (Kaggle `city_day.csv`)

In [ ]:
def synthetic_india_city_day():
    cities = ["Delhi", "Lucknow", "Kolkata", "Patna", "Ahmedabad", "Mumbai", "Chennai", "Bengaluru",
              "Hyderabad", "Jaipur", "Amritsar", "Chandigarh", "Gurugram", "Visakhapatnam", "Bhopal"]
    base = dict(zip(cities, [115, 105, 80, 110, 70, 50, 40, 40, 50, 75, 80, 70, 100, 55, 65]))
    dates = pd.date_range("2015-01-01", "2020-07-01", freq="D")
    month_f = {1: 1.6, 2: 1.2, 3: 0.9, 4: 0.85, 5: 0.9, 6: 0.7, 7: 0.45, 8: 0.4, 9: 0.55, 10: 1.0, 11: 1.7, 12: 1.9}
    rows = []
    for c in cities:
        f = np.array([month_f[m] for m in dates.month])
        pm25 = base[c] * f * np.exp(rng.normal(0, 0.35, len(dates)))
        rows.append(pd.DataFrame({
            "City": c, "Date": dates, "PM2.5": pm25, "PM10": pm25 * rng.uniform(1.5, 2.2, len(dates)),
            "NO": rng.gamma(2, 6, len(dates)), "NO2": 20 + 0.15 * pm25 + rng.normal(0, 8, len(dates)),
            "NOx": 30 + 0.2 * pm25 + rng.normal(0, 10, len(dates)), "NH3": rng.gamma(3, 6, len(dates)),
            "CO": np.abs(0.6 + 0.008 * pm25 + rng.normal(0, 0.3, len(dates))),
            "SO2": np.abs(8 + 0.03 * pm25 + rng.normal(0, 3, len(dates))),
            "O3": np.abs(35 + rng.normal(0, 15, len(dates))),
            "AQI": np.clip(pm25 * 1.6 + rng.normal(0, 15, len(dates)), 10, 500)}))
    df = pd.concat(rows, ignore_index=True)
    for c in ["PM2.5", "PM10", "NO2", "SO2", "O3", "CO"]:
        df.loc[rng.random(len(df)) < 0.12, c] = np.nan
    return df.sample(n=29531, random_state=SEED).sort_values(["City", "Date"]).reset_index(drop=True)


f2 = next(iter(sorted(RAW.glob("city_day*.csv"))), None)
if f2 is not None:
    india = pd.read_csv(f2, parse_dates=["Date"])
    ds2_mode = f"REAL Kaggle file ({f2.name})"
else:
    india = synthetic_india_city_day()
    ds2_mode = "FALLBACK synthetic (city_day format)"
keep = [c for c in ["City", "Date", "PM2.5", "PM10", "NO2", "SO2", "O3", "CO", "AQI"] if c in india.columns]
india = india[keep]
india.to_csv(PROC / "india_city_day_standardized.csv", index=False)
print(f"Dataset 2 [{ds2_mode}]: {len(india):,} city-day records, {india.City.nunique()} cities")
summary["dataset_2_india_air_quality"] = {"records": int(len(india)), "mode": ds2_mode}

## Dataset 1 - Global Air Pollution Dataset (Kaggle)

In [ ]:
def synthetic_global_air():
    n = 23500
    pm = np.clip(rng.lognormal(3.6, 0.9, n), 1, 500)
    df = pd.DataFrame({
        "Country": rng.choice(["Pakistan", "India", "China", "USA", "Brazil", "Germany", "Egypt", "Nigeria", "Japan", "Iran"], n),
        "City": [f"City_{i}" for i in rng.integers(0, 4000, n)],
        "PM2.5 AQI Value": pm.round(),
        "CO AQI Value": np.clip(rng.gamma(1.5, 1.2, n), 0, 100).round(),
        "Ozone AQI Value": np.clip(rng.normal(38, 22, n), 0, 250).round(),
        "NO2 AQI Value": np.clip(rng.gamma(1.4, 2.0, n), 0, 100).round()})
    df["AQI Value"] = df[["PM2.5 AQI Value", "CO AQI Value", "Ozone AQI Value", "NO2 AQI Value"]].max(axis=1)
    df["AQI Category"] = pd.cut(df["AQI Value"], [-1, 50, 100, 150, 200, 300, 1000],
                                labels=["Good", "Moderate", "Unhealthy for Sensitive Groups", "Unhealthy", "Very Unhealthy", "Hazardous"])
    return df


f1 = next(iter(sorted(RAW.glob("*global*pollution*.csv"))), None)
if f1 is not None:
    glob_air = pd.read_csv(f1)
    ds1_mode = f"REAL Kaggle file ({f1.name})"
else:
    glob_air = synthetic_global_air()
    ds1_mode = "FALLBACK synthetic (global pollution format)"
glob_air.to_csv(PROC / "global_air_pollution_standardized.csv", index=False)
print(f"Dataset 1 [{ds1_mode}]: {len(glob_air):,} records")
summary["dataset_1_global_air_pollution"] = {"records": int(len(glob_air)), "mode": ds1_mode}

## Dataset 3 - Asthma Disease Dataset (Kaggle)
The real Kaggle file is small (a few thousand patients). If it has fewer than 10,000 rows, the pool is
topped up by bootstrap-with-jitter and the extra rows are flagged `is_augmented = 1`.

In [ ]:
CLIN_COLS = ["Age", "BMI", "Smoking", "PhysicalActivity", "PollutionExposure", "DustExposure", "PetAllergy",
             "FamilyHistoryAsthma", "HistoryOfAllergies", "LungFunctionFEV1", "Wheezing", "ShortnessOfBreath",
             "ChestTightness", "Coughing", "NighttimeSymptoms", "ExerciseInduced"]


def synthetic_patients(n=10200):
    age = rng.integers(5, 82, n)
    lat = rng.normal(0, 1, n)
    df = pd.DataFrame({
        "Age": age, "Gender": rng.integers(0, 2, n),
        "BMI": np.clip(rng.normal(26.5, 6, n), 14, 45),
        "Smoking": ((rng.random(n) < 0.16) & (age > 17)).astype(int),
        "PhysicalActivity": rng.uniform(0, 10, n), "PollutionExposure": rng.uniform(0, 10, n),
        "DustExposure": rng.uniform(0, 10, n), "PetAllergy": (rng.random(n) < 0.2).astype(int),
        "FamilyHistoryAsthma": (rng.random(n) < 0.25).astype(int), "HistoryOfAllergies": (rng.random(n) < 0.3).astype(int),
        "LungFunctionFEV1": np.clip(3.3 - 0.018 * np.abs(age - 30) + rng.normal(0, 0.55, n) - 0.3 * lat * 0, 1.0, 4.2)})
    for c, w in [("Wheezing", .6), ("ShortnessOfBreath", .6), ("ChestTightness", .5), ("Coughing", .5),
                 ("NighttimeSymptoms", .4), ("ExerciseInduced", .4)]:
        p = 1 / (1 + np.exp(-(-0.9 + w * lat + 0.4 * df["FamilyHistoryAsthma"] + 0.3 * df["Smoking"])))
        df[c] = (rng.random(n) < p).astype(int)
    df["Diagnosis"] = (df[["Wheezing", "ShortnessOfBreath", "ChestTightness", "Coughing"]].sum(axis=1) + rng.normal(0, .8, n) > 2).astype(int)
    for c in ["BMI", "LungFunctionFEV1", "PollutionExposure", "DustExposure", "PhysicalActivity"]:
        df.loc[rng.random(n) < 0.03, c] = np.nan  # missing clinical entries
    return df


f3 = next(iter(sorted(RAW.glob("asthma*.csv"))), None)
patients_real_rows = 0
if f3 is not None:
    pat = pd.read_csv(f3)
    patients_real_rows = len(pat)
    for c in CLIN_COLS:
        if c not in pat.columns:
            pat[c] = np.nan
    if "Gender" not in pat.columns:
        pat["Gender"] = np.nan
    if "Diagnosis" not in pat.columns:
        pat["Diagnosis"] = np.nan
    pat = pat[["Age", "Gender"] + [c for c in CLIN_COLS if c != "Age"] + ["Diagnosis"]].copy()
    pat["is_augmented"] = 0
    empty_cols = [c for c in CLIN_COLS if pat[c].isna().all()]
    if empty_cols:  # e.g. a different asthma CSV that lacks some clinical fields
        print(f"[WARN] Columns missing in the real asthma file, filled from the fallback generator: {empty_cols}")
        filler = synthetic_patients(len(pat))
        for c in empty_cols:
            pat[c] = filler[c].values
    ds3_mode = f"REAL Kaggle file ({f3.name}), {patients_real_rows:,} rows"
    if len(pat) < 10000:
        extra = pat.sample(n=10000 - len(pat), replace=True, random_state=SEED).copy()
        extra["Age"] = np.clip(extra["Age"] + rng.integers(-2, 3, len(extra)), 1, 90)
        extra["BMI"] = np.clip(extra["BMI"] + rng.normal(0, 0.8, len(extra)), 12, 50)
        extra["LungFunctionFEV1"] = np.clip(extra["LungFunctionFEV1"] + rng.normal(0, 0.08, len(extra)), 0.8, 4.5)
        extra["is_augmented"] = 1
        pat = pd.concat([pat, extra], ignore_index=True)
        print(f"[WARN] Real asthma file has {patients_real_rows:,} rows (<10,000). Added {len(extra):,} bootstrap+jitter rows (is_augmented=1).")
else:
    pat = synthetic_patients()
    pat["is_augmented"] = 0
    ds3_mode = "FALLBACK synthetic (asthma dataset format)"
pat.insert(0, "patient_id", np.arange(len(pat)))
pat.to_csv(PROC / "patients_pool.csv", index=False)
print(f"Dataset 3 [{ds3_mode}]: {len(pat):,} patient records")
summary["dataset_3_asthma"] = {"records": int(len(pat)), "mode": ds3_mode, "real_rows": int(patients_real_rows)}

## Join strategy (spatiotemporal + clinical risk tier)
1. **Spatial key.** Each patient gets a home city from the five Open-Meteo cities (population-weighted draw, stored once per patient).
2. **Temporal key.** A patient-day record is created by pairing a patient with a random hour from *their* city's Open-Meteo timeline (rounded to the hour).
3. **Environmental context.** The exact pollutant + weather values at that city-hour are attached. Datasets 1 and 2 supply the AQI reference table and the multi-year seasonal baseline used in the EDA.
4. **Clinical risk tier.** Patient clinical profile + ambient exposure produce a four-level label (Low / Moderate / High / Critical).

**Important:** no public dataset links a person's daily pollution exposure to an asthma attack. The label is therefore a documented
*proxy* built from epidemiological relationships (PM2.5, gases, weather, age, smoking, FEV1, symptoms, allergy) plus noise.

In [ ]:
N_RECORDS, N_DUP = 15000, 75
city_names = list(CITIES)
city_w = np.array([CITIES[c][4] for c in city_names])
pat["city"] = rng.choice(city_names, size=len(pat), p=city_w / city_w.sum())

pick = rng.integers(0, len(pat), N_RECORDS)
recs = pat.iloc[pick].reset_index(drop=True).copy()
stamps = []
for city in city_names:
    tl = timeline[timeline.city == city].sort_values("timestamp").reset_index(drop=True)
    tl = tl.iloc[24:]  # keep rows that have >= 24h history for lag/rolling features
    m = (recs["city"] == city).values
    stamps.append(pd.Series(tl["timestamp"].values[rng.integers(0, len(tl), m.sum())], index=recs.index[m]))
recs["timestamp"] = pd.concat(stamps).sort_index()
env_cols = ["pm2_5", "pm10", "no2", "so2", "o3", "co", "temp", "humidity", "wind"]
merged = recs.merge(timeline[["city", "timestamp"] + env_cols], on=["city", "timestamp"], how="left")

## Label creation (proxy asthma-risk tier)

In [ ]:
def make_risk_label(df):
    d = df.copy()
    for c in env_cols + CLIN_COLS:
        d[c] = d[c].fillna(d[c].median())  # only for scoring; the real table keeps its gaps
    pm = d.pm2_5.clip(0, 400) / 60
    gas = 0.5 * d.no2.clip(0, 300) / 80 + 0.5 * d.so2.clip(0, 200) / 40
    stagn = (d.humidity / 100) / (d.wind + 1)
    age_risk = ((d.Age <= 12) | (d.Age >= 60)).astype(float)
    clinical = (0.9 * d.FamilyHistoryAsthma + 0.8 * d.Smoking + 0.6 * d.HistoryOfAllergies + 0.35 * d.PetAllergy
                + 0.12 * d.DustExposure + 0.12 * d.PollutionExposure + 0.9 * age_risk
                + 0.8 * (4 - d.LungFunctionFEV1).clip(0, 3) / 3 + 0.35 * d.Wheezing + 0.35 * d.ShortnessOfBreath
                + 0.3 * d.ChestTightness + 0.2 * d.Coughing + 0.35 * d.NighttimeSymptoms + 0.4 * d.ExerciseInduced)
    score = (1.6 * pm + 0.5 * gas + 0.25 * d.o3 / 100 + 0.35 * stagn * 10 + clinical
             + 0.8 * pm * (age_risk + d.FamilyHistoryAsthma) + 0.04 * d.PhysicalActivity * pm
             + rng.normal(0, 0.6, len(d)))
    q = np.quantile(score, [0.35, 0.68, 0.90])
    return np.digitize(score, q)


merged["Asthma_Risk_Level"] = make_risk_label(merged)
merged["Asthma_Risk_Label"] = merged["Asthma_Risk_Level"].map({0: "Low", 1: "Moderate", 2: "High", 3: "Critical"})
merged.insert(0, "record_id", np.arange(len(merged)))

# repeated submissions / feed retries: a few exact duplicates so the cleaning step has real work to do
dups = merged.sample(N_DUP, random_state=SEED)
merged = pd.concat([merged, dups], ignore_index=True).sample(frac=1, random_state=SEED).reset_index(drop=True)
merged = merged.drop(columns=["Diagnosis"], errors="ignore")
merged.to_csv(PROC / "merged_breathesafe_dataset.csv", index=False)
print(f"Merged dataset: {merged.shape[0]:,} rows x {merged.shape[1]} cols -> data/processed/merged_breathesafe_dataset.csv")
print(merged["Asthma_Risk_Label"].value_counts(normalize=True).round(3).to_string())

## Documentation: sources, bias, join strategy

In [ ]:
summary["merged_rows"] = int(len(merged))
summary["target_distribution"] = merged["Asthma_Risk_Label"].value_counts().to_dict()
with open(METRICS / "dataset_summary.json", "w") as fh:
    json.dump(summary, fh, indent=2)

doc = f"""# Data sources, bias and join strategy

## Sources
| # | Dataset | URL | Records used | Mode this run |
|---|---------|-----|-------------:|---------------|
| 1 | Global Air Pollution Dataset | https://www.kaggle.com/datasets/hasibalmuzdadid/global-air-pollution-dataset | {summary['dataset_1_global_air_pollution']['records']:,} | {ds1_mode} |
| 2 | Air Quality Data in India | https://www.kaggle.com/datasets/rohanrao/air-quality-data-in-india | {summary['dataset_2_india_air_quality']['records']:,} | {ds2_mode} |
| 3 | Asthma Disease Dataset | https://www.kaggle.com/datasets/rabieelkharoua/asthma-disease-dataset | {summary['dataset_3_asthma']['records']:,} | {ds3_mode} |
| 4 | Open-Meteo Air Quality + Weather API | https://air-quality-api.open-meteo.com/v1/air-quality , https://api.open-meteo.com/v1/forecast | {summary['dataset_4_open_meteo']['records']:,} | {ds4_mode} |

## Known biases
- **Urban sensor skew:** monitoring stations sit mostly in large cities; rural exposure is under-represented.
- **Winter smog seasonality:** PM2.5 peaks October-January in the Indo-Gangetic plain; a three-month window over-weights one season.
- **Self-reported clinical bias:** symptoms, exposure and allergy fields are self-reported and can be under- or over-stated.
- **Cross-country transfer:** the Indian and global datasets are used for seasonal context, not as a direct measurement of Pakistani exposure.
- **Proxy label:** the risk tier is rule-based plus noise, not a recorded clinical outcome.

## Join strategy
Spatial key (city) + temporal key (hour) link a patient-day to the Open-Meteo timeline; Datasets 1-2 provide AQI reference and seasonal baselines;
the clinical risk tier combines patient profile and ambient exposure. See the notebook markdown above for the full description.
"""
(METRICS / "data_sources_bias_join.md").write_text(doc, encoding="utf-8")
print("Saved outputs/metrics/dataset_summary.json and data_sources_bias_join.md")

# 02 - Cleaning, Imputation, Outliers, Feature Engineering, Feature Selection, EDA plots
Input: `data/processed/merged_breathesafe_dataset.csv` + `environment_timeline.csv` (from script 01)
Output: `data/processed/engineered_features.csv`, `outputs/plots/fig01..fig08`, `outputs/metrics/*`
(Figures 09-12 need trained models, so script 03 saves them.)

In [ ]:
import json
import sys
import warnings
from functools import partial
from pathlib import Path

import matplotlib
if "ipykernel" not in sys.modules:
    matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import RFE, SelectKBest, f_classif, mutual_info_classif
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
try:
    HERE = Path(__file__).resolve().parent
except NameError:
    HERE = Path.cwd()
ROOT = HERE.parent if HERE.name == "code" else HERE
PROC, PLOTS, METRICS = ROOT / "data/processed", ROOT / "outputs/plots", ROOT / "outputs/metrics"
for p in (PROC, PLOTS, METRICS):
    p.mkdir(parents=True, exist_ok=True)
SEED = 42
sns.set_theme(style="whitegrid", context="notebook")
CLASS_NAMES = ["Low", "Moderate", "High", "Critical"]
interp = {}   # figure -> written interpretation


def save_fig(fig, name):
    fig.tight_layout()
    fig.savefig(PLOTS / name, dpi=200, bbox_inches="tight")
    if "ipykernel" in sys.modules:
        plt.show()
    plt.close(fig)

## 1. Load + data cleaning (duplicates, timestamp alignment, invalid sensor values)

In [ ]:
merged = pd.read_csv(PROC / "merged_breathesafe_dataset.csv", parse_dates=["timestamp"])
timeline = pd.read_csv(PROC / "environment_timeline.csv", parse_dates=["timestamp"])
log = {"rows_raw": int(len(merged))}

merged = merged.drop_duplicates().reset_index(drop=True)
log["duplicates_removed"] = log["rows_raw"] - len(merged)

# timestamp alignment: everything on the hour, one reading per city-hour
merged["timestamp"] = merged["timestamp"].dt.floor("h")
timeline["timestamp"] = timeline["timestamp"].dt.floor("h")
n0 = len(timeline)
timeline = timeline.drop_duplicates(["city", "timestamp"]).sort_values(["city", "timestamp"]).reset_index(drop=True)
log["timeline_duplicate_hours_removed"] = n0 - len(timeline)

POLL = ["pm2_5", "pm10", "no2", "so2", "o3", "co"]
ENV = POLL + ["temp", "humidity", "wind"]
invalid = {}
for c in POLL + ["wind"]:
    bad = timeline[c] < 0
    invalid[c] = int(bad.sum())
    timeline.loc[bad, c] = np.nan
bad = (timeline["humidity"] < 0) | (timeline["humidity"] > 100)
invalid["humidity"] = int(bad.sum())
timeline.loc[bad, "humidity"] = np.nan
log["invalid_sensor_values_set_to_nan"] = invalid

## 2. Environmental sensors: time-series interpolation
Short gaps (<= 6 h) are interpolated in time; longer gaps are filled from the nearest valid reading.

In [ ]:
log["timeline_missing_before"] = {c: int(timeline[c].isna().sum()) for c in ENV}
parts = []
for city, g in timeline.groupby("city"):
    g = g.set_index("timestamp").sort_index()
    g[ENV] = g[ENV].interpolate(method="time", limit=6, limit_direction="both").ffill().bfill()
    parts.append(g.reset_index())
timeline = pd.concat(parts, ignore_index=True)
log["timeline_missing_after"] = {c: int(timeline[c].isna().sum()) for c in ENV}
pm25_before = timeline["pm2_5"].copy()   # saved for the before/after plot

## 3. Outlier handling: IQR winsorization
Pollutants use a wide fence (Q3 + 3 x IQR, floor at 0) so real severe smog episodes survive; only sensor-spike
territory is capped. Weather variables use the usual 1.5 x IQR fence on both sides.

In [ ]:
def winsorize(s, k, lower=True):
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    lo = q1 - k * iqr if lower else 0
    hi = q3 + k * iqr
    return s.clip(lo, hi), int(((s < lo) | (s > hi)).sum()), float(hi)


capped = {}
for c in POLL:
    timeline[c], capped[c], hi = winsorize(timeline[c], 3.0, lower=False)
    capped[c] = {"n_capped": capped[c], "upper_fence": round(hi, 2)}
for c in ["temp", "humidity", "wind"]:
    timeline[c], n, hi = winsorize(timeline[c], 1.5)
    capped[c] = {"n_capped": n, "upper_fence": round(hi, 2)}
timeline["humidity"] = timeline["humidity"].clip(0, 100)
log["environment_capped"] = capped

## 4. Time-based features from the cleaned timeline (rolling 24 h, 1-day lag, daily temperature range)

In [ ]:
g = timeline.groupby("city")
timeline["pm25_roll24"] = g["pm2_5"].transform(lambda s: s.rolling(24, min_periods=12).mean())
timeline["pm25_lag1d"] = g["pm2_5"].shift(24)
timeline["temp_range24"] = g["temp"].transform(lambda s: s.rolling(24, min_periods=12).max() - s.rolling(24, min_periods=12).min())
for c in ["pm25_roll24", "pm25_lag1d", "temp_range24"]:
    timeline[c] = timeline.groupby("city")[c].transform(lambda s: s.bfill())

# weekly city trend for the dashboard / fig03
weekly = (timeline.set_index("timestamp").groupby("city")["pm2_5"].resample("W").mean().reset_index())
weekly.to_csv(METRICS / "city_pm25_weekly.csv", index=False)

# attach cleaned environment to patient-day records (replaces the raw readings)
merged = merged.drop(columns=ENV)
merged = merged.merge(timeline[["city", "timestamp"] + ENV + ["pm25_roll24", "pm25_lag1d", "temp_range24"]],
                      on=["city", "timestamp"], how="left")
log["rows_without_environment_match"] = int(merged["pm2_5"].isna().sum())
merged = merged.dropna(subset=["pm2_5"]).reset_index(drop=True)

## 5. Clinical columns: range checks, KNN imputation (numeric), mode imputation (binary)

In [ ]:
NUM_CLIN = ["Age", "BMI", "PhysicalActivity", "PollutionExposure", "DustExposure", "LungFunctionFEV1"]
BIN_CLIN = ["Smoking", "PetAllergy", "FamilyHistoryAsthma", "HistoryOfAllergies", "Wheezing", "ShortnessOfBreath",
            "ChestTightness", "Coughing", "NighttimeSymptoms", "ExerciseInduced"]
ranges = {"Age": (1, 100), "BMI": (10, 60), "PhysicalActivity": (0, 10), "PollutionExposure": (0, 10),
          "DustExposure": (0, 10), "LungFunctionFEV1": (0.5, 6)}
for c, (lo, hi) in ranges.items():
    merged.loc[(merged[c] < lo) | (merged[c] > hi), c] = np.nan
log["clinical_missing_before"] = {c: int(merged[c].isna().sum()) for c in NUM_CLIN + BIN_CLIN}

scaler = StandardScaler()
scaled = scaler.fit_transform(merged[NUM_CLIN])          # NaNs are ignored by StandardScaler
scaled = KNNImputer(n_neighbors=5, weights="distance").fit_transform(scaled)
merged[NUM_CLIN] = scaler.inverse_transform(scaled)
merged[BIN_CLIN] = SimpleImputer(strategy="most_frequent").fit_transform(merged[BIN_CLIN]).astype(int)
for c in ["BMI", "LungFunctionFEV1"]:
    merged[c], n, hi = winsorize(merged[c], 1.5)
log["clinical_missing_after"] = {c: int(merged[c].isna().sum()) for c in NUM_CLIN + BIN_CLIN}
log["rows_clean"] = int(len(merged))

## 6. Twelve engineered features
`add_engineered_features` is copied unchanged into `app.py`, so training and the web app use identical formulas.

In [ ]:
def add_engineered_features(df):
    d = df.copy()
    d["PM25_to_PM10_Ratio"] = (d["pm2_5"] / (d["pm10"] + 1e-6)).clip(0, 1.5)
    d["Rolling_24h_PM25"] = d["pm25_roll24"]
    d["PM25_Lag_1d"] = d["pm25_lag1d"]
    d["Smog_Severity_Index"] = 100 * (0.45 * d["pm2_5"] / 60 + 0.20 * d["pm10"] / 120 + 0.20 * d["no2"] / 80 + 0.15 * d["so2"] / 40)
    d["Temp_Humidity_Discomfort_Index"] = d["temp"] - 0.55 * (1 - d["humidity"] / 100) * (d["temp"] - 14.5)
    d["Atmospheric_Stagnation_Index"] = (d["humidity"] / 100) / (d["wind"] + 1.0)
    n, s = d["no2"] / 80, d["so2"] / 40
    d["NO2_SO2_Synergy_Score"] = n + s + 0.5 * n * s
    d["Diurnal_Temp_Range"] = d["temp_range24"]
    d["Clinical_Vulnerability_Score"] = (0.25 * d["FamilyHistoryAsthma"] + 0.20 * d["Smoking"] + 0.20 * d["HistoryOfAllergies"]
                                         + 0.10 * d["PetAllergy"] + 0.15 * d["DustExposure"] / 10
                                         + 0.10 * ((4.0 - d["LungFunctionFEV1"]).clip(0, 3) / 3))
    d["Pollution_x_Vulnerability"] = d["pm2_5"] * d["Clinical_Vulnerability_Score"]
    d["Pediatric_Geriatric_Risk_Flag"] = ((d["Age"] <= 12) | (d["Age"] >= 60)).astype(int)
    d["Outdoor_Exposure_Risk_Score"] = d["PhysicalActivity"] * d["Smog_Severity_Index"] / 10
    return d


ENGINEERED = ["PM25_to_PM10_Ratio", "Rolling_24h_PM25", "PM25_Lag_1d", "Smog_Severity_Index",
              "Temp_Humidity_Discomfort_Index", "Atmospheric_Stagnation_Index", "NO2_SO2_Synergy_Score",
              "Diurnal_Temp_Range", "Clinical_Vulnerability_Score", "Pollution_x_Vulnerability",
              "Pediatric_Geriatric_Risk_Flag", "Outdoor_Exposure_Risk_Score"]
RAW_FEATURES = ["pm2_5", "pm10", "no2", "so2", "o3", "temp", "humidity", "wind", "Age", "BMI", "Smoking",
                "PhysicalActivity", "PollutionExposure", "DustExposure", "LungFunctionFEV1", "Wheezing",
                "ShortnessOfBreath", "ChestTightness", "Coughing", "NighttimeSymptoms", "ExerciseInduced",
                "PetAllergy", "FamilyHistoryAsthma", "HistoryOfAllergies"]
CANDIDATES = RAW_FEATURES + ENGINEERED

data = add_engineered_features(merged)
assert data[CANDIDATES].isna().sum().sum() == 0, "NaN left in candidate features"
y = data["Asthma_Risk_Level"].astype(int)
X = data[CANDIDATES]
print(f"Clean rows: {len(data):,} | candidate features: {len(CANDIDATES)} ({len(ENGINEERED)} engineered)")

## 7. Feature selection (Mutual Information / ANOVA, Random-Forest Gini, RFE)

In [ ]:
# Method 1: SelectKBest with Mutual Information (ANOVA F-test computed alongside)
mi_sel = SelectKBest(partial(mutual_info_classif, random_state=SEED), k=20).fit(X, y)
f_sel = SelectKBest(f_classif, k=20).fit(X, y)
sel = pd.DataFrame({"feature": CANDIDATES, "MI": mi_sel.scores_, "ANOVA_F": f_sel.scores_})

# Method 2: Random-Forest Gini importance + Recursive Feature Elimination
sub = data.sample(8000, random_state=SEED).index
rf_full = RandomForestClassifier(n_estimators=200, class_weight="balanced", max_depth=12, n_jobs=-1, random_state=SEED).fit(X, y)
sel["RF_Gini"] = rf_full.feature_importances_
rfe = RFE(RandomForestClassifier(n_estimators=80, max_depth=10, class_weight="balanced", n_jobs=-1, random_state=SEED),
          n_features_to_select=20, step=2).fit(X.loc[sub], y.loc[sub])
sel["RFE_Rank"] = rfe.ranking_
sel["RFE_Selected"] = rfe.support_

# combine: average of the three ranks (1 = best); keep the top 20
sel["rank_MI"] = sel["MI"].rank(ascending=False)
sel["rank_Gini"] = sel["RF_Gini"].rank(ascending=False)
sel["avg_rank"] = (sel["rank_MI"] + sel["rank_Gini"] + sel["RFE_Rank"]) / 3
sel["is_engineered"] = sel["feature"].isin(ENGINEERED)
sel = sel.sort_values("avg_rank").reset_index(drop=True)
SELECTED = sel["feature"].head(20).tolist()
sel.to_csv(METRICS / "feature_selection_scores.csv", index=False)
print("Selected features:", SELECTED)

## 8. Save engineered dataset + selection metadata

In [ ]:
out = data[["record_id", "patient_id", "city", "timestamp"] + CANDIDATES + ["Asthma_Risk_Level", "Asthma_Risk_Label"]]
out.to_csv(PROC / "engineered_features.csv", index=False)
meta = {"candidates": CANDIDATES, "engineered": ENGINEERED, "selected": SELECTED, "class_names": CLASS_NAMES,
        "defaults": {c: float(data[c].median()) for c in CANDIDATES}}
(METRICS / "selected_features.json").write_text(json.dumps(meta, indent=2))
(METRICS / "cleaning_log.json").write_text(json.dumps(log, indent=2))
print(f"Saved engineered_features.csv {out.shape}")

## 9. Plots 1-8 with written interpretation

In [ ]:
# Fig 1 - target class distribution
cnt = y.map(dict(enumerate(CLASS_NAMES))).value_counts().reindex(CLASS_NAMES)
fig, ax = plt.subplots(figsize=(7, 4.5))
bars = ax.bar(CLASS_NAMES, cnt.values, color=["#2a9d8f", "#e9c46a", "#f4a261", "#d62828"])
for b, v in zip(bars, cnt.values):
    ax.text(b.get_x() + b.get_width() / 2, v, f"{v:,}\n({v / len(y):.1%})", ha="center", va="bottom")
ax.set(title="Fig 1 - Asthma risk class distribution", ylabel="Patient-day records", ylim=(0, cnt.max() * 1.2))
save_fig(fig, "fig01_target_class_distribution.png")
interp["fig01"] = (f"The four risk tiers are imbalanced on purpose: Low {cnt['Low'] / len(y):.0%}, Moderate {cnt['Moderate'] / len(y):.0%}, "
                   f"High {cnt['High'] / len(y):.0%}, Critical {cnt['Critical'] / len(y):.0%}. Critical cases are the rarest but the most dangerous, "
                   "so plain accuracy would hide poor performance on them. This is why class weights and recall-focused metrics are used later.")

# Fig 2 - pollutant correlation heatmap
fig, ax = plt.subplots(figsize=(7.5, 6))
corr = timeline[ENV].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0, ax=ax, square=True, cbar_kws={"shrink": .8})
ax.set_title("Fig 2 - Pollutant & weather correlation (Open-Meteo timeline)")
save_fig(fig, "fig02_pollutant_correlation_heatmap.png")
interp["fig02"] = (f"PM2.5 and PM10 move together (r = {corr.loc['pm2_5', 'pm10']:.2f}), which is expected because both come from the same combustion, dust and "
                   f"smog sources. PM2.5 also correlates with NO2 (r = {corr.loc['pm2_5', 'no2']:.2f}) and CO (r = {corr.loc['pm2_5', 'co']:.2f}), pointing to traffic and burning. "
                   f"Humidity and wind show weaker links (humidity r = {corr.loc['pm2_5', 'humidity']:.2f}, wind r = {corr.loc['pm2_5', 'wind']:.2f}). "
                   "The strong pollutant-pollutant correlation is a warning about redundancy, which is why feature selection follows.")

# Fig 3 - seasonal / city comparison
india = pd.read_csv(PROC / "india_city_day_standardized.csv", parse_dates=["Date"]).dropna(subset=["PM2.5"])
india["Month"] = india["Date"].dt.month
top = india.groupby("City")["PM2.5"].mean().nlargest(6).index
monthly = india[india.City.isin(top)].groupby(["City", "Month"])["PM2.5"].mean().unstack(0)
monthly.to_csv(METRICS / "india_monthly_pm25.csv")
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
monthly.plot(ax=axes[0], marker="o")
axes[0].set(title="South Asian cities: monthly mean PM2.5 (Dataset 2)", xlabel="Month", ylabel="PM2.5 (ug/m3)")
axes[0].set_xticks(range(1, 13))
for city, gg in weekly.groupby("city"):
    axes[1].plot(gg["timestamp"], gg["pm2_5"], marker=".", label=city)
axes[1].set(title="Pakistani cities: weekly mean PM2.5 (Dataset 4)", xlabel="Week", ylabel="PM2.5 (ug/m3)")
axes[1].legend()
axes[1].tick_params(axis="x", rotation=30)
fig.suptitle("Fig 3 - Seasonal and city smog comparison")
save_fig(fig, "fig03_seasonal_city_smog_comparison.png")
peak_m = int(monthly.mean(axis=1).idxmax())
low_m = int(monthly.mean(axis=1).idxmin())
worst_pk = weekly.groupby("city")["pm2_5"].mean().idxmax()
wk_first = weekly.groupby("city")["pm2_5"].apply(lambda s: s.head(4).mean())
wk_last = weekly.groupby("city")["pm2_5"].apply(lambda s: s.tail(4).mean())
trend = "rises" if wk_last.mean() > wk_first.mean() else "falls"
interp["fig03"] = (f"In the multi-year city-day data the six most polluted cities peak in month {peak_m} (mean {monthly.mean(axis=1).max():.0f} ug/m3) and bottom out in month "
                   f"{low_m} ({monthly.mean(axis=1).min():.0f} ug/m3), the winter-smog versus monsoon-wash pattern. In the Pakistani timeline, {worst_pk} has the highest average PM2.5, "
                   f"and the average of the five cities {trend} from {wk_first.mean():.0f} ug/m3 (first 4 weeks) to {wk_last.mean():.0f} ug/m3 (last 4 weeks). "
                   "A three-month window covers only part of the smog season, a bias noted in the report.")

# Fig 4 - PM2.5 vs risk
d4 = data.assign(Risk=data["Asthma_Risk_Label"])
fig, ax = plt.subplots(figsize=(7.5, 4.8))
sns.boxplot(data=d4, x="Risk", y="pm2_5", order=CLASS_NAMES, palette=["#2a9d8f", "#e9c46a", "#f4a261", "#d62828"], showfliers=False, ax=ax)
ax.set(title="Fig 4 - Ambient PM2.5 by asthma risk tier", ylabel="PM2.5 (ug/m3)", xlabel="")
save_fig(fig, "fig04_pm25_vs_asthma_risk_boxplot.png")
med = d4.groupby("Risk")["pm2_5"].median().reindex(CLASS_NAMES)
interp["fig04"] = (f"Median PM2.5 rises from {med['Low']:.0f} ug/m3 (Low) to {med['Moderate']:.0f} (Moderate), {med['High']:.0f} (High) and {med['Critical']:.0f} (Critical). "
                   "The boxes overlap, so PM2.5 alone cannot separate the tiers: a person with a poor clinical profile can reach a high tier on a moderately polluted day. "
                   "That overlap is the reason a model combining pollution with the patient profile is needed instead of a plain AQI reading.")

# Fig 5 - outliers before / after
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
bins = np.linspace(0, max(pm25_before.quantile(0.999), timeline["pm2_5"].max()), 60)
axes[0].hist(pm25_before, bins=bins, color="#e76f51")
axes[0].set(title="Before winsorization", xlabel="PM2.5 (ug/m3)", ylabel="Hourly readings")
axes[1].hist(timeline["pm2_5"], bins=bins, color="#2a9d8f")
axes[1].set(title="After winsorization (3 x IQR fence)", xlabel="PM2.5 (ug/m3)")
fig.suptitle("Fig 5 - PM2.5 outlier handling")
save_fig(fig, "fig05_outlier_before_after.png")
interp["fig05"] = (f"Before capping the maximum hourly PM2.5 was {pm25_before.max():.0f} ug/m3; after capping it is {timeline['pm2_5'].max():.0f} ug/m3 "
                   f"(fence at {capped['pm2_5']['upper_fence']} ug/m3, {capped['pm2_5']['n_capped']} readings changed out of {len(timeline):,}). "
                   "The wide 3 x IQR fence leaves the long right tail of real smog episodes intact and only trims isolated sensor-style spikes.")

# Fig 6 - engineered feature correlations
fig, ax = plt.subplots(figsize=(10, 8.5))
ec = data[ENGINEERED + ["Asthma_Risk_Level"]].corr()
sns.heatmap(ec, annot=True, fmt=".2f", cmap="RdBu_r", center=0, ax=ax, annot_kws={"size": 7}, cbar_kws={"shrink": .8})
ax.set_title("Fig 6 - Engineered features and risk level correlation")
save_fig(fig, "fig06_engineered_feature_correlations.png")
tc = ec["Asthma_Risk_Level"].drop("Asthma_Risk_Level").abs().sort_values(ascending=False)
interp["fig06"] = (f"The engineered features most correlated with the risk level are {tc.index[0]} (|r| = {tc.iloc[0]:.2f}), {tc.index[1]} ({tc.iloc[1]:.2f}) and {tc.index[2]} ({tc.iloc[2]:.2f}). "
                   f"The weakest is {tc.index[-1]} ({tc.iloc[-1]:.2f}). Several pollution features are strongly inter-correlated (Smog Severity, Rolling 24h, Lag 1d), so they carry overlapping information; "
                   "tree models cope with this, but the SVM and logistic regression benefit from the later feature selection.")

# Fig 7 - Mutual information
s7 = sel.sort_values("MI").tail(20)
fig, ax = plt.subplots(figsize=(8, 7))
ax.barh(s7["feature"], s7["MI"], color=np.where(s7["is_engineered"], "#264653", "#8ab17d"))
ax.set(title="Fig 7 - Mutual Information scores (top 20)", xlabel="Mutual information")
ax.legend(handles=[plt.Rectangle((0, 0), 1, 1, color="#264653"), plt.Rectangle((0, 0), 1, 1, color="#8ab17d")], labels=["Engineered", "Raw"], loc="lower right")
save_fig(fig, "fig07_mutual_information_scores.png")
n_eng_top = int(s7["is_engineered"].sum())
interp["fig07"] = (f"{n_eng_top} of the top 20 mutual-information features are engineered ones. The top three are {', '.join(sel.sort_values('MI', ascending=False)['feature'].head(3))}. "
                   "Mutual information captures non-linear dependence on the risk tier, so combined indices such as Pollution_x_Vulnerability rank high because they merge environment and patient factors.")

# Fig 8 - RFE ranking + RF Gini
fig, axes = plt.subplots(1, 2, figsize=(14, 7))
s8 = sel.sort_values("RFE_Rank", ascending=False)
axes[0].barh(s8["feature"], s8["RFE_Rank"], color=np.where(s8["RFE_Selected"], "#2a9d8f", "#bbbbbb"))
axes[0].set(title="RFE rank (1 = kept)", xlabel="Rank")
s8b = sel.sort_values("RF_Gini")
axes[1].barh(s8b["feature"], s8b["RF_Gini"], color="#e76f51")
axes[1].set(title="Random Forest Gini importance", xlabel="Importance")
fig.suptitle("Fig 8 - RFE ranking and Random Forest importance")
save_fig(fig, "fig08_rfe_and_rf_importance.png")
agree = len(set(sel[sel.RFE_Selected].feature) & set(sel.sort_values("MI", ascending=False).feature.head(20)))
interp["fig08"] = (f"RFE kept 20 of {len(CANDIDATES)} features, and {agree} of them also appear in the mutual-information top 20, so the two methods largely agree. "
                   f"The Random Forest ranks {sel.sort_values('RF_Gini', ascending=False).feature.iloc[0]} first. Final features are the 20 with the best average rank across MI, Gini and RFE.")

In [ ]:
(PLOTS / "figure_interpretations.json").write_text(json.dumps(interp, indent=2))
for k, v in interp.items():
    print(f"[{k}] {v}\n")

# 03 - Train, Tune, Evaluate and Compare the Models
Models: Random Forest, Gradient Boosting (XGBoost if installed, else scikit-learn GradientBoosting), SVM (RBF),
plus Logistic Regression as a baseline. Input: `data/processed/engineered_features.csv`.

In [ ]:
import copy
import json
import sys
import time
import warnings
from pathlib import Path

import joblib
import matplotlib
if "ipykernel" not in sys.modules:
    matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats
from sklearn.base import clone
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, auc, classification_report, confusion_matrix, f1_score,
                             precision_recall_curve, precision_score, recall_score, roc_auc_score, roc_curve)
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.svm import SVC
from sklearn.utils.class_weight import compute_sample_weight

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except Exception:
    HAS_XGB = False

warnings.filterwarnings("ignore")
try:
    HERE = Path(__file__).resolve().parent
except NameError:
    HERE = Path.cwd()
ROOT = HERE.parent if HERE.name == "code" else HERE
PROC, PLOTS, METRICS, MODELS = ROOT / "data/processed", ROOT / "outputs/plots", ROOT / "outputs/metrics", ROOT / "models"
for p in (PLOTS, METRICS, MODELS):
    p.mkdir(parents=True, exist_ok=True)
SEED = 42
sns.set_theme(style="whitegrid", context="notebook")
CLASS_NAMES = ["Low", "Moderate", "High", "Critical"]
interp = json.loads((PLOTS / "figure_interpretations.json").read_text()) if (PLOTS / "figure_interpretations.json").exists() else {}


def save_fig(fig, name):
    fig.tight_layout()
    fig.savefig(PLOTS / name, dpi=200, bbox_inches="tight")
    if "ipykernel" in sys.modules:
        plt.show()
    plt.close(fig)

## 1. Stratified train / validation / test split (70 / 15 / 15)

In [ ]:
meta = json.loads((METRICS / "selected_features.json").read_text())
FEATURES = meta["selected"]
df = pd.read_csv(PROC / "engineered_features.csv")
X, y = df[FEATURES], df["Asthma_Risk_Level"].astype(int)
X_tr, X_tmp, y_tr, y_tmp = train_test_split(X, y, test_size=0.30, stratify=y, random_state=SEED)
X_val, X_te, y_val, y_te = train_test_split(X_tmp, y_tmp, test_size=0.50, stratify=y_tmp, random_state=SEED)
print(f"train {len(X_tr):,} | validation {len(X_val):,} | test {len(X_te):,} | features {len(FEATURES)}")
print("Class balance (train):", y_tr.value_counts(normalize=True).sort_index().round(3).to_dict())

# class imbalance: 'balanced' class weights (RF, SVM, LR) / balanced sample weights (boosting)
w_tr = compute_sample_weight("balanced", y_tr)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

## 2. Models and hyper-parameter search spaces
Tuning uses `RandomizedSearchCV` with 5-fold stratified CV (scoring = macro F1) on a stratified sample of up to 5,000
training rows, which keeps the SVM tractable. The winning parameters are then refit on the full training set.

In [ ]:
if HAS_XGB:
    gb_name = "Gradient Boosting (XGBoost)"
    gb_clf = XGBClassifier(objective="multi:softprob", eval_metric="mlogloss", n_jobs=-1, random_state=SEED, tree_method="hist")
    gb_grid = {"clf__n_estimators": [150, 250, 400], "clf__learning_rate": [0.05, 0.1, 0.2], "clf__max_depth": [3, 4, 6],
               "clf__subsample": [0.8, 1.0], "clf__colsample_bytree": [0.7, 1.0]}
else:
    gb_name = "Gradient Boosting (sklearn)"
    gb_clf = GradientBoostingClassifier(random_state=SEED)
    gb_grid = {"clf__n_estimators": [100, 200], "clf__learning_rate": [0.05, 0.1, 0.2], "clf__max_depth": [2, 3, 4],
               "clf__subsample": [0.8, 1.0]}
print("Boosting model:", gb_name)

SPECS = {
    "Random Forest": (RandomForestClassifier(class_weight="balanced", n_jobs=-1, random_state=SEED),
                      {"clf__n_estimators": [150, 300], "clf__max_depth": [8, 12, 16], "clf__min_samples_leaf": [1, 3, 5],
                       "clf__max_features": ["sqrt", 0.5]}, False),
    gb_name: (gb_clf, gb_grid, True),
    "SVM (RBF)": (SVC(kernel="rbf", probability=False, class_weight="balanced", random_state=SEED),
                  {"clf__C": [0.5, 1, 3, 10], "clf__gamma": ["scale", 0.01, 0.05, 0.1]}, False),
    "Logistic Regression (baseline)": (LogisticRegression(class_weight="balanced", max_iter=3000),
                                       {"clf__C": [0.01, 0.1, 1, 10]}, False),
}

## 3. Tune, refit, time and evaluate

In [ ]:
def single_row_latency_ms(model, Xdf, n=100):
    rows = [Xdf.iloc[[i]] for i in range(min(n, len(Xdf)))]
    t0 = time.perf_counter()
    for r in rows:
        model.predict_proba(r)
    return (time.perf_counter() - t0) / len(rows) * 1000


sub_idx = X_tr.sample(min(5000, len(X_tr)), random_state=SEED).index
fitted, results, tuned = {}, [], {}
for name, (clf, grid, uses_sw) in SPECS.items():
    print(f"\n=== {name} ===")
    pipe = Pipeline([("scaler", StandardScaler()), ("clf", clf)])
    search = RandomizedSearchCV(pipe, grid, n_iter=6, cv=cv, scoring="f1_macro", n_jobs=-1, random_state=SEED, refit=False)
    fit_kw = {"clf__sample_weight": compute_sample_weight("balanced", y_tr.loc[sub_idx])} if uses_sw else {}
    search.fit(X_tr.loc[sub_idx], y_tr.loc[sub_idx], **fit_kw)
    best = {k.replace("clf__", ""): v for k, v in search.best_params_.items()}
    tuned[name] = {"best_params": best, "tuning_cv_f1_macro": round(float(search.best_score_), 4)}
    print("best params:", best, "| tuning CV F1:", round(search.best_score_, 4))

    final = Pipeline([("scaler", StandardScaler()), ("clf", clone(clf).set_params(**best))])
    if isinstance(clf, SVC):
        cv_model = clone(final)                                # SVM scored via decision values in CV (faster)
        final.set_params(clf__probability=True)                # probabilities needed for ROC-AUC and the app
    else:
        cv_model = clone(final)
    cv_kw = {"params": {"clf__sample_weight": w_tr}} if uses_sw else {}
    cv_scores = cross_val_score(cv_model, X_tr, y_tr, cv=cv, scoring="f1_macro", n_jobs=1, **cv_kw)

    t0 = time.perf_counter()
    final.fit(X_tr, y_tr, **({"clf__sample_weight": w_tr} if uses_sw else {}))
    train_s = time.perf_counter() - t0
    fitted[name] = final

    def evaluate(Xs, ys):
        proba = final.predict_proba(Xs)
        pred = proba.argmax(axis=1)
        return proba, pred, {
            "Accuracy": accuracy_score(ys, pred),
            "Precision": precision_score(ys, pred, average="macro"),
            "Recall": recall_score(ys, pred, average="macro"),
            "F1": f1_score(ys, pred, average="macro"),
            "ROC_AUC": roc_auc_score(ys, proba, multi_class="ovr", average="macro")}

    _, _, val_m = evaluate(X_val, y_val)
    proba_te, pred_te, te_m = evaluate(X_te, y_te)
    hr_true, hr_pred = y_te >= 2, pred_te >= 2
    results.append({
        "Model": name, **{k: round(v, 4) for k, v in te_m.items()},
        "HighRisk_Recall": round(recall_score(hr_true, hr_pred), 4),
        "Val_F1": round(val_m["F1"], 4), "Val_Accuracy": round(val_m["Accuracy"], 4),
        "CV5_F1_mean": round(cv_scores.mean(), 4), "CV5_F1_std": round(cv_scores.std(), 4),
        "Train_Time_s": round(train_s, 2), "Inference_ms_per_pred": round(single_row_latency_ms(final, X_te), 3)})
    tuned[name]["cv_scores"] = [round(float(s), 4) for s in cv_scores]
    tuned[name]["proba_te"], tuned[name]["pred_te"] = proba_te, pred_te
    print(results[-1])

res = pd.DataFrame(results)
res.to_csv(METRICS / "model_comparison.csv", index=False)
print("\n", res.drop(columns=["Val_Accuracy"]).to_string(index=False))

## 4. Pick the best model (validation macro F1) and the two models to compare statistically

In [ ]:
core = res[~res.Model.str.contains("baseline")].sort_values("Val_F1", ascending=False).reset_index(drop=True)
best_name, second_name = core.loc[0, "Model"], core.loc[1, "Model"]
print(f"Best on validation F1: {best_name} | runner-up: {second_name}")

# 5-fold CV paired t-test (same folds for both models)
a, b = np.array(tuned[best_name]["cv_scores"]), np.array(tuned[second_name]["cv_scores"])
t_stat, t_p = stats.ttest_rel(a, b)

# McNemar test on the test set
ca, cb = tuned[best_name]["pred_te"] == y_te.values, tuned[second_name]["pred_te"] == y_te.values
n01, n10 = int((ca & ~cb).sum()), int((~ca & cb).sum())     # best right / second wrong, and the reverse
if n01 + n10 == 0:
    mc_stat, mc_p, mc_kind = 0.0, 1.0, "no discordant pairs"
elif n01 + n10 < 25:
    mc_stat, mc_p, mc_kind = float(min(n01, n10)), float(stats.binomtest(min(n01, n10), n01 + n10, 0.5).pvalue), "exact binomial"
else:
    mc_stat = (abs(n01 - n10) - 1) ** 2 / (n01 + n10)
    mc_p, mc_kind = float(stats.chi2.sf(mc_stat, 1)), "chi-square with continuity correction"
tests = {
    "compared": [best_name, second_name],
    "paired_t_test_5fold_cv": {"fold_scores_best": a.tolist(), "fold_scores_second": b.tolist(), "t_statistic": round(float(t_stat), 4),
                               "p_value": round(float(t_p), 4), "significant_at_0.05": bool(t_p < 0.05)},
    "mcnemar_test_holdout": {"best_right_second_wrong": n01, "best_wrong_second_right": n10, "test_type": mc_kind,
                             "statistic": round(float(mc_stat), 4), "p_value": round(mc_p, 4), "significant_at_0.05": bool(mc_p < 0.05)}}
(METRICS / "statistical_tests.json").write_text(json.dumps(tests, indent=2))
print(json.dumps(tests, indent=2))

## 5. Error analysis (best model on the test set)

In [ ]:
best_pred, best_proba = tuned[best_name]["pred_te"], tuned[best_name]["proba_te"]
cm = confusion_matrix(y_te, best_pred)
(METRICS / "classification_report_best.txt").write_text(classification_report(y_te, best_pred, target_names=CLASS_NAMES, digits=3))
print(classification_report(y_te, best_pred, target_names=CLASS_NAMES, digits=3))

true_hi = (y_te >= 2).values
under = true_hi & (best_pred < 2)                                  # High/Critical predicted as Low/Moderate = dangerous false negatives
crit_missed = (y_te.values == 3) & (best_pred < 2)
over = (y_te.values < 2) & (best_pred >= 2)
wrong = best_pred != y_te.values
prof = X_te.copy()
prof["_wrong"] = wrong
z = (prof[prof._wrong][FEATURES].mean() - prof[~prof._wrong][FEATURES].mean()) / prof[FEATURES].std()
profile = pd.DataFrame({"mean_misclassified": prof[prof._wrong][FEATURES].mean(), "mean_correct": prof[~prof._wrong][FEATURES].mean(),
                        "standardized_difference": z}).sort_values("standardized_difference", key=abs, ascending=False)
profile.to_csv(METRICS / "misclassified_feature_profile.csv")
fn_profile = X_te[under][FEATURES].mean()
error = {
    "best_model": best_name, "confusion_matrix": cm.tolist(), "n_test": int(len(y_te)), "n_misclassified": int(wrong.sum()),
    "misclassification_rate": round(float(wrong.mean()), 4),
    "dangerous_false_negatives_high_or_critical_predicted_low_or_moderate": int(under.sum()),
    "high_risk_false_negative_rate": round(float(under.sum() / max(true_hi.sum(), 1)), 4),
    "critical_predicted_low_or_moderate": int(crit_missed.sum()),
    "false_alarms_low_or_moderate_predicted_high_or_critical": int(over.sum()),
    "per_class_error_rate": {CLASS_NAMES[i]: round(float(1 - cm[i, i] / cm[i].sum()), 4) for i in range(4)},
    "top_features_differing_in_errors": profile.head(5).round(3)["standardized_difference"].to_dict(),
    "mean_pm25_missed_high_risk_cases": round(float(X_te[under]["pm2_5"].mean()), 2) if under.any() and "pm2_5" in FEATURES else None,
    "mean_pm25_correctly_flagged_high_risk": round(float(X_te[true_hi & ~under]["pm2_5"].mean()), 2) if "pm2_5" in FEATURES else None}
(METRICS / "error_analysis.json").write_text(json.dumps(error, indent=2))
print(json.dumps(error, indent=2))

# high-risk alert threshold: smallest-miss rule used by the web app
p_hi = best_proba[:, 2:].sum(axis=1)
sweep = []
for th in np.arange(0.15, 0.71, 0.05):
    pr = p_hi >= th
    sweep.append({"threshold": round(float(th), 2), "recall_high_risk": round(float(recall_score(true_hi, pr)), 4),
                  "precision_high_risk": round(float(precision_score(true_hi, pr, zero_division=0)), 4),
                  "false_alarms": int((pr & ~true_hi).sum())})
sweep = pd.DataFrame(sweep)
sweep.to_csv(METRICS / "threshold_sweep.csv", index=False)
ok = sweep[sweep.recall_high_risk >= 0.90]
ALERT_TH = float(ok.threshold.max()) if len(ok) else 0.35
print(f"High-risk alert threshold chosen: {ALERT_TH} (largest threshold with recall >= 0.90)")

## 6. Figures 9-12

In [ ]:
Y_bin = label_binarize(y_te, classes=[0, 1, 2, 3])
colors = dict(zip(SPECS, ["#2a9d8f", "#e76f51", "#264653", "#999999"]))

# Fig 9 - ROC
fig, axes = plt.subplots(1, 2, figsize=(13, 5.2))
for name in SPECS:
    fpr, tpr, _ = roc_curve(Y_bin.ravel(), tuned[name]["proba_te"].ravel())
    axes[0].plot(fpr, tpr, color=colors[name], label=f"{name} (AUC {auc(fpr, tpr):.3f})", ls="--" if "baseline" in name else "-")
axes[0].plot([0, 1], [0, 1], "k:", lw=1)
axes[0].set(title="Micro-average ROC, all models", xlabel="False positive rate", ylabel="True positive rate")
axes[0].legend(loc="lower right", fontsize=8)
for i, cn in enumerate(CLASS_NAMES):
    fpr, tpr, _ = roc_curve(Y_bin[:, i], best_proba[:, i])
    axes[1].plot(fpr, tpr, label=f"{cn} (AUC {auc(fpr, tpr):.3f})")
axes[1].plot([0, 1], [0, 1], "k:", lw=1)
axes[1].set(title=f"Per-class ROC - {best_name}", xlabel="False positive rate")
axes[1].legend(loc="lower right")
fig.suptitle("Fig 9 - ROC curves")
save_fig(fig, "fig09_roc_curves.png")
pc_auc = {cn: roc_auc_score(Y_bin[:, i], best_proba[:, i]) for i, cn in enumerate(CLASS_NAMES)}
interp["fig09"] = (f"All three main models reach a macro ROC-AUC between {res[~res.Model.str.contains('baseline')].ROC_AUC.min():.3f} and {res[~res.Model.str.contains('baseline')].ROC_AUC.max():.3f}; "
                   f"the logistic-regression baseline scores {res[res.Model.str.contains('baseline')].ROC_AUC.iloc[0]:.3f}. For {best_name}, per-class AUC is "
                   + ", ".join(f"{k} {v:.3f}" for k, v in pc_auc.items()) + ". The middle tiers (Moderate, High) are hardest because they sit between neighbours, while the extremes separate cleanly.")

# Fig 10 - Precision-Recall
fig, axes = plt.subplots(1, 2, figsize=(13, 5.2))
for name in SPECS:
    p_, r_, _ = precision_recall_curve(Y_bin.ravel(), tuned[name]["proba_te"].ravel())
    axes[0].plot(r_, p_, color=colors[name], label=f"{name} (AUC {auc(r_, p_):.3f})", ls="--" if "baseline" in name else "-")
axes[0].set(title="Micro-average precision-recall, all models", xlabel="Recall", ylabel="Precision")
axes[0].legend(loc="lower left", fontsize=8)
for i, cn in enumerate(CLASS_NAMES):
    p_, r_, _ = precision_recall_curve(Y_bin[:, i], best_proba[:, i])
    axes[1].plot(r_, p_, label=f"{cn} (AUC {auc(r_, p_):.3f})")
axes[1].set(title=f"Per-class precision-recall - {best_name}", xlabel="Recall")
axes[1].legend(loc="lower left")
fig.suptitle("Fig 10 - Precision-Recall curves")
save_fig(fig, "fig10_precision_recall_curves.png")
crit_pr = auc(*precision_recall_curve(Y_bin[:, 3], best_proba[:, 3])[1::-1])
interp["fig10"] = (f"Precision-recall curves matter more than ROC here because Critical cases are only about {Y_bin[:, 3].mean():.0%} of the test set. "
                   f"For {best_name} the Critical class has a PR-AUC of {crit_pr:.3f}. The curves show how much precision is given up when the alert threshold is lowered to catch more high-risk patients, "
                   f"which is the trade-off behind the app's alert threshold of {ALERT_TH:.2f}.")

# Fig 11 - confusion matrices
show = [n for n in SPECS if "baseline" not in n]
fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))
for ax, name in zip(axes, show):
    c = confusion_matrix(y_te, tuned[name]["pred_te"])
    sns.heatmap(c / c.sum(axis=1, keepdims=True), annot=np.array([[f"{v}\n({v / c[i].sum():.0%})" for v in row] for i, row in enumerate(c)]),
                fmt="", cmap="Blues", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, cbar=False, ax=ax)
    ax.set(title=name, xlabel="Predicted", ylabel="Actual")
fig.suptitle("Fig 11 - Confusion matrices (test set, row-normalised)")
save_fig(fig, "fig11_confusion_matrices.png")
interp["fig11"] = (f"For {best_name}, {error['n_misclassified']} of {error['n_test']} test records are misclassified ({error['misclassification_rate']:.1%}). Nearly all errors are one tier away from the truth. "
                   f"{error['dangerous_false_negatives_high_or_critical_predicted_low_or_moderate']} High/Critical patients were predicted Low/Moderate (a high-risk miss rate of {error['high_risk_false_negative_rate']:.1%}), "
                   f"and {error['false_alarms_low_or_moderate_predicted_high_or_critical']} Low/Moderate patients were flagged High/Critical.")

# Fig 12 - latency vs accuracy
fig, ax = plt.subplots(figsize=(8, 5.5))
for _, r in res.iterrows():
    ax.scatter(r.Inference_ms_per_pred, r.Accuracy, s=120 + r.Train_Time_s * 12, color=colors[r.Model], alpha=.8, edgecolor="k")
    ax.annotate(f"{r.Model}\n(train {r.Train_Time_s:.1f}s)", (r.Inference_ms_per_pred, r.Accuracy), textcoords="offset points", xytext=(8, 6), fontsize=8)
ax.set(title="Fig 12 - Latency vs accuracy (bubble size = training time)", xlabel="Inference time per prediction (ms, single row)", ylabel="Test accuracy")
ax.margins(0.25)
save_fig(fig, "fig12_latency_vs_accuracy.png")
fast, slow = res.loc[res.Inference_ms_per_pred.idxmin()], res.loc[res.Inference_ms_per_pred.idxmax()]
interp["fig12"] = (f"{fast.Model} answers fastest ({fast.Inference_ms_per_pred:.2f} ms per single prediction) and {slow.Model} slowest ({slow.Inference_ms_per_pred:.2f} ms). "
                   f"Accuracy differs by only {res.Accuracy.max() - res.Accuracy.min():.3f} between all four models, so for a web app every model is fast enough; "
                   "the choice should be driven by recall and stability rather than latency.")

(PLOTS / "figure_interpretations.json").write_text(json.dumps(interp, indent=2))
(PLOTS / "figure_interpretations.md").write_text("# Figure interpretations\n\n" + "\n\n".join(f"**{k.upper()}** - {v}" for k, v in sorted(interp.items())))

## 7. Save the artefacts (pipelines, scaler, metadata)

In [ ]:
file_names = {"Random Forest": "random_forest_pipeline.joblib", gb_name: "gradient_boosting_pipeline.joblib",
              "SVM (RBF)": "svm_pipeline.joblib", "Logistic Regression (baseline)": "logistic_regression_pipeline.joblib"}
for name, model in fitted.items():
    joblib.dump(model, MODELS / file_names[name])
joblib.dump(fitted[best_name], MODELS / "best_model.joblib")
joblib.dump(StandardScaler().fit(X_tr), MODELS / "scaler.joblib")   # standalone copy; each pipeline also embeds its own scaler
model_meta = {"features": FEATURES, "class_names": CLASS_NAMES, "best_model": best_name, "alert_threshold": ALERT_TH,
              "model_files": file_names, "uses_xgboost": HAS_XGB, "metrics": res.to_dict(orient="records"),
              "best_params": {k: v["best_params"] for k, v in tuned.items()}, "defaults": meta["defaults"]}
(MODELS / "model_meta.json").write_text(json.dumps(model_meta, indent=2))

top2 = res.set_index("Model")
summary_md = f"""# Evaluation summary
Best model (validation macro-F1): **{best_name}**. Runner-up: {second_name}.

{res.drop(columns=['Val_Accuracy']).to_markdown(index=False) if hasattr(res, 'to_markdown') and 'tabulate' in sys.modules else res.to_string(index=False)}

Paired t-test (5-fold CV): t = {tests['paired_t_test_5fold_cv']['t_statistic']}, p = {tests['paired_t_test_5fold_cv']['p_value']}
McNemar ({mc_kind}): p = {tests['mcnemar_test_holdout']['p_value']} (b = {n01}, c = {n10})
High-risk false negatives: {error['dangerous_false_negatives_high_or_critical_predicted_low_or_moderate']} ({error['high_risk_false_negative_rate']:.1%})
"""
(METRICS / "evaluation_summary.md").write_text(summary_md)
print(summary_md)
print("Saved models/, outputs/metrics/, outputs/plots/fig09-fig12")

## 8. Live test prediction (one hand-made scenario)

In [ ]:
demo = X_te.iloc[[0]].copy()
for k, v in {"pm2_5": 180.0, "Age": 67}.items():
    if k in demo.columns:
        demo[k] = v
proba = fitted[best_name].predict_proba(demo)[0]
print("Test scenario ->", dict(zip(CLASS_NAMES, proba.round(3))), "| predicted:", CLASS_NAMES[int(proba.argmax())])

## Final: try your own scenario
Edit the numbers below and run the cell.

In [ ]:
scenario = X_te.iloc[[0]].copy()
scenario['pm2_5'] = 220.0   # try 40, 120, 220 ...
scenario['Age'] = 8         # try 25, 70 ...
p = fitted[best_name].predict_proba(scenario)[0]
print(dict(zip(CLASS_NAMES, p.round(3))), '->', CLASS_NAMES[int(p.argmax())])